In [1]:
import os
import json
import numpy as np
from pathlib import Path
from typing import List, Dict
from textwrap import dedent
from langchain_openai import AzureChatOpenAI
from langchain.vectorstores.faiss import FAISS
from langchain_openai.embeddings import AzureOpenAIEmbeddings
from tqdm import tqdm

In [2]:
root = Path().absolute().parent

config_dir = root / "src/data"
vectorstores_dir = root / "src/vectorstores"

In [3]:
llm = AzureChatOpenAI(
    azure_deployment="gpt4o",
    openai_api_type="azure",
    openai_api_version="2024-02-01",
    api_key=os.getenv("OPENAI_API_KEY"),
    azure_endpoint=os.getenv("OPENAI_ENDPOINT"),
    temperature=0.75,
)

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-large",
    api_version="2024-02-01",
    api_key=os.getenv("OPENAI_API_KEY"),
    azure_endpoint=os.getenv("OPENAI_ENDPOINT"),
)

In [4]:
jira_raw_pasd = json.load(open(config_dir / "jira_pasd.json","r"))
confluence_raw_arb = json.load(open(config_dir / "confluence_arb.json","r"))
confluence_raw_ap = json.load(open(config_dir / "confluence_ap.json","r"))
confluence_raw_po = json.load(open(config_dir / "confluence_po.json","r"))

In [5]:
def create_jira_documents(jira_contents: Dict) -> List:
    prompts = []
    
    for content in jira_contents:
        if content["content"]['num_comments'] > 5:
            prompt = dedent(
                f"""\
                    ### START OF JIRA ISSUE:

                    ## JIRA Ticket URL: {content['url']}
                    ## Issue Summary: {content['content']['Summary']}
                    ## Issue Type: {content['content']['Issue Type']}
                    ## Issue Raised By (Creator): {content['content']['Creator']}
                    ## Issue Description: {content['content']['Description']}
                    ## Expected Behaviour by the Creator: {content['content']['Custom field (Expected Behavior)']}
                    ## Steps to Reproduce Issue (if applicable): {content['content']['Custom field (Steps To Reproduce)']}
                    ## Conversation:\n {content['content']['comments']}

                    ### END OF JIRA ISSUE"""
            )
            prompts.append(prompt)

    return prompts

In [6]:
def create_confluence_documents(confluence_contents):
    texts = []
    
    for content in confluence_contents:
        url = content['url']
        title = content['content']['title']
        body = content['content']['body']
        ancestor_pages = list(content['content']['ancestors'].values()) + [title]
        ancestors = " / ".join(ancestor_pages)

        text = dedent(
            f"""\
### START OF CONFLUENCE PAGE
Confluence Page URL: {url}

Navigation: {ancestors}
# {title}
{body}

### END OF CONFLUENCE PAGE"""
        )
        texts.append(text)
    
    return texts


In [7]:
texts_arb = create_confluence_documents(confluence_raw_arb)
texts_ap = create_confluence_documents(confluence_raw_ap)
texts_po = create_confluence_documents(confluence_raw_po)
texts_pasd = create_jira_documents(jira_raw_pasd)

print(len(texts_arb))
print(len(texts_ap))
print(len(texts_po))
print(len(texts_pasd))

618
2197
254
848


In [10]:
def create_vectorstore(texts: List):
    vectorstores = []
    batch_size = 100
    max_idx = int(np.ceil(len(texts)/batch_size) * batch_size)

    # create embeddings vectorstores in batch sizes of 100
    for index in tqdm(range(0,max_idx,batch_size)):
        vectorstore = FAISS.from_texts(
            texts=texts[index : index + batch_size],
            embedding=embeddings,
        )
        vectorstores.append(vectorstore)
    
    # merge all vectorstores
    for vectorstore in vectorstores[1:]:
        vectorstores[0].merge_from(vectorstore)

    # return the merged vectorstore
    return vectorstores[0]

# create vectorstores
vectorstore_arb = create_vectorstore(texts_arb)
vectorstore_ap = create_vectorstore(texts_ap)
vectorstore_po = create_vectorstore(texts_po)
vectorstore_pasd = create_vectorstore(texts_pasd)

# save vectorstores
os.chdir(vectorstores_dir)

vectorstore_arb.save_local(f"vectorstore_arb")
vectorstore_ap.save_local(f"vectorstore_ap")
vectorstore_po.save_local(f"vectorstore_po")
vectorstore_pasd.save_local(f"vectorstore_pasd")

100%|██████████| 3/3 [00:17<00:00,  5.69s/it]


In [11]:
# Load Vectorstores
vectorstore_arb = FAISS.load_local("vectorstore_arb", embeddings, allow_dangerous_deserialization=True)
vectorstore_ap = FAISS.load_local("vectorstore_ap", embeddings, allow_dangerous_deserialization=True)
vectorstore_po = FAISS.load_local("vectorstore_po", embeddings, allow_dangerous_deserialization=True)
vectorstore_pasd = FAISS.load_local("vectorstore_pasd", embeddings, allow_dangerous_deserialization=True)

In [12]:
vectorstore_pasd.merge_from(vectorstore_ap)
vectorstore_pasd.merge_from(vectorstore_po)
vectorstore_pasd.merge_from(vectorstore_arb)

In [16]:
vectorstore_pasd.index.ntotal

3917

In [14]:
vectorstore_pasd.save_local("vectorstore_master")